<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix D · Change of Numeraire and Forward Measures

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_d_change_of_numeraire_forward_measures.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Change of numeraire is the bookkeeping device behind most rates-derivative pricing shortcuts. After working through this companion, you can:

- explain why each strictly positive traded numeraire has an associated martingale measure;
- derive T-forward probabilities from real-world probabilities and a state discount kernel on a two-state example;
- price a payoff paid at a fixed maturity as a zero-coupon bond price times its expectation under the matching forward measure;
- compute a simple forward rate from two discount factors with the accrual fraction stated, as in caplet, floorlet, and swaption pricing.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

Chapter 14 prices caplets and floorlets by discounting an expectation under the T-forward measure, and Chapter 13 treats forward rates as martingales under their own measures. This appendix supplies the formal reason those shortcuts are arbitrage-free: relative prices are martingales only under the measure associated with the chosen numeraire, and the discount factor moves outside the expectation only when the bond maturity matches the payoff date.

The companion figure script `appx_d_forward_measure_martingale.py` shows the payoff of that choice: under the forward measure matching the payoff date, the relevant forward quantity stays centered on its current level instead of carrying an avoidable drift term.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_d_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

- Write the change-of-numeraire pricing formula for a payoff X_T paid at T under the numeraire P(t,T), and state which quantity is a martingale under the T-forward measure.
- Compute the forward rate f(0;T1,T2) from P(0,T1)=0.9600 and P(0,T2)=0.9200 with accrual fraction 1, and state the simple-compounding convention embedded in the formula.
- For the two-state caplet in the appendix, identify the measure of the probabilities 0.40 and 0.60, and explain why the price is the bond price times the expected payoff rather than the expectation of a discounted payoff.
- Explain why a swaption is priced most naturally under an annuity measure rather than a single-maturity forward measure.

## Mini-lab: Forward price as a forward-measure expectation

The mini-lab isolates the change-of-numeraire identity on a two-state example. A traded asset pays 1.06 in the high state and 0.99 in the low state at maturity T. Real-world probabilities are 0.60 and 0.40, and the state discount kernel is 0.40 and 0.55 per unit of probability. The probability-weighted kernel sums to the zero-coupon bond price P(0,T)=0.4600, which serves as the numeraire. All values are illustrative decimals; no compounding convention is needed because discounting enters through the kernel itself.

In [ ]:
import numpy as np
import pandas as pd

probability = np.array([0.60, 0.40])  # real-world state probabilities
kernel = np.array([0.40, 0.55])  # state discount kernel per unit prob
payoff = np.array([1.06, 0.99])  # asset payoff at maturity T

bond = np.dot(probability, kernel)  # zero-coupon bond price P(0, T)
forward_prob = probability * kernel / bond  # T-forward measure weights
forward_price = np.dot(forward_prob, payoff)  # forward price
value = bond * forward_price  # price as bond times forward mean
check = np.dot(probability, kernel * payoff)  # direct check
pd.Series({"P(0,T)": bond, "q_H": forward_prob[0],
           "F_0": forward_price, "V_0": value, "check": check})

### What to observe

The T-forward probabilities are 0.521739 and 0.478261, so the forward price is 0.521739 times 1.06 plus 0.478261 times 0.99, or 1.026522. Multiplying by P(0,T)=0.4600 gives the value 0.472200, identical to the direct state-pricing calculation up to floating-point error. Reweighting the probabilities by the kernel relative to the bond price is the entire measure change.

## Exercises

1. **Reprice the two-state caplet.** With accrual fraction 1, strike 4%, P(0,T2)=0.9200, and T2-forward probabilities 0.40 for a 6% fixing and 0.60 for a 3% fixing, compute the caplet value per unit of notional. *Hint:* The expected payoff is 0.0080 and the value is 0.9200 times that amount.
2. **Change the kernel.** In the mini-lab, set the low-state kernel to 0.50 and recompute the forward probabilities. State which state gains weight and why. *Hint:* The bond price falls to 0.4400 and the high-state forward probability rises above 0.52.
3. **Forward rates from discount factors.** Compute the one-period forward rate from P(0,T1)=0.9600 and P(0,T2)=0.9200 with accrual fraction 1, and explain why it is a curve-implied breakeven rate rather than a forecast. *Hint:* The rate is 4.3478% under the simple-compounding convention embedded in 1 + alpha times f.
4. **Annuity numeraire.** Explain which quantity becomes a martingale when the present value of a swap's fixed leg is used as the numeraire, and which instrument this measure prices most naturally. *Hint:* The forward swap rate is a martingale under the annuity measure, the standard device for swaptions.
5. **Payoff-date mismatch.** Suppose the caplet payoff were paid at T3 rather than T2. State which bond appears in the pricing formula and what extra expectation is required. *Hint:* Under the T3-forward measure, the T2-maturity payoff must be discounted with a forward discount factor inside the expectation.

For more detail, see Appendix D of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">